# Last.fm batch recommendations

Run strategy **(b) random track** over `albums_2plus.csv`. Results are written back into the same file (one row per album).

Re-run the batch cell to resume — rows with `status` already set are skipped.

In [13]:
import importlib
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

import lastfm_albums
from lastfm_albums import DEFAULT_FETCH_FLOOR, recommend_album, set_request_delay

In [17]:
DATA_DIR = Path("../datasets")
WORK_PATH = DATA_DIR / "albums_2plus.csv"

TRACK_SELECTION = "random"  # top_listener | random | all_tracks_overlap
N_RECS = 1
FETCH_FLOOR = DEFAULT_FETCH_FLOOR
REQUEST_DELAY_SEC = 1
RANDOM_SEED = 42
SAVE_EVERY = 10
MAX_ALBUMS = None  # e.g. 10 for a dry run

In [18]:
def rec_columns(n_recs):
    cols = []
    for rank in range(1, n_recs + 1):
        cols.extend([
            f"rec_{rank}_album",
            f"rec_{rank}_artist",
            f"rec_{rank}_score",
        ])
    return cols


def is_pending(status):
    return pd.isna(status) or str(status).strip() == ""


EXTRA_COLS = ["status", "error", "seed_track", *rec_columns(N_RECS)]

work = pd.read_csv(WORK_PATH)
for col in EXTRA_COLS:
    if col not in work.columns:
        work[col] = pd.NA

# avoid float64 columns when CSV has only NaN (breaks writing error strings)
work["status"] = work["status"].astype("string")
work["error"] = work["error"].astype("string")
work["seed_track"] = work["seed_track"].astype("string")
for col in rec_columns(N_RECS):
    if col.endswith("_score"):
        work[col] = pd.to_numeric(work[col], errors="coerce")
    else:
        work[col] = work[col].astype("string")

pending = work.index[work["status"].map(is_pending)]
if MAX_ALBUMS is not None:
    pending = pending[:MAX_ALBUMS]

print(f"Work file: {WORK_PATH}")
print(f"Total albums: {len(work):,} | pending: {len(pending):,} | done: {len(work) - len(pending):,}")
work.head()

Work file: ../datasets/albums_2plus.csv
Total albums: 2,498 | pending: 1,838 | done: 660


,album_id,artist,album,review_count,status,error,seed_track,rec_1_album,rec_1_artist,rec_1_score,rec_1_url
0,3,!!!,Myth Takes,2,ok,<NA>,Heart of Hearts,Pieces Of The People We Love,The Rapture,0.219018,NaN
1,5,!!!,"Strange Weather, Isn't It?",2,ok,<NA>,Hollow,Donkey,Cansei de Ser Sexy,0.644824,NaN
2,57,1990s,Cookies,2,ok,<NA>,You're Supposed To Be My Friend,Little Death,Pete and the Pirates,0.293653,NaN
3,58,1990s,Kicks,2,ok,<NA>,I Don't Even Know What That Is,"Right Thoughts, Right Words, Right Action (Del...",Franz Ferdinand,0.371913,NaN
4,85,2562,Unbalance,2,ok,<NA>,Flashback,Great Lengths,Martyn,0.313865,NaN


In [19]:
set_request_delay(REQUEST_DELAY_SEC)

since_save = 0
for idx in tqdm(pending, desc="Last.fm batch"):
    row = work.loc[idx]
    album_seed = RANDOM_SEED + int(row["album_id"])

    try:
        _, seed_track, recs, _ = recommend_album(
            row["album"],
            artist=row["artist"],
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=TRACK_SELECTION,
            random_seed=album_seed,
        )
        if recs.empty:
            work.loc[idx, "status"] = "no_results"
            work.loc[idx, "error"] = pd.NA
        else:
            work.loc[idx, "status"] = "ok"
            work.loc[idx, "error"] = pd.NA
            work.loc[idx, "seed_track"] = seed_track["name"] if seed_track else pd.NA
            for rank in range(1, N_RECS + 1):
                work.loc[idx, f"rec_{rank}_album"] = pd.NA
                work.loc[idx, f"rec_{rank}_artist"] = pd.NA
                work.loc[idx, f"rec_{rank}_score"] = pd.NA
            for rank, (_, rec) in enumerate(recs.iterrows(), start=1):
                if rank > N_RECS:
                    break
                work.loc[idx, f"rec_{rank}_album"] = rec["album"]
                work.loc[idx, f"rec_{rank}_artist"] = rec["artist"]
                work.loc[idx, f"rec_{rank}_score"] = rec["similarity_score"]
    except Exception as exc:
        work.loc[idx, "status"] = "error"
        work.loc[idx, "error"] = str(exc)[:500]

    since_save += 1
    if since_save >= SAVE_EVERY:
        work.to_csv(WORK_PATH, index=False)
        since_save = 0

work.to_csv(WORK_PATH, index=False)
print("Saved.")

Last.fm batch:   0%|          | 0/1838 [00:00<?, ?it/s]

Saved.


In [21]:
print(work["status"].value_counts(dropna=False))
work.loc[work["status"] == "ok", ["artist", "album", "seed_track", "rec_1_album", "rec_1_artist"]].head(10)

status
ok            2132
no_results     300
error           66
Name: count, dtype: int64[pyarrow]


,artist,album,seed_track,rec_1_album,rec_1_artist
0,!!!,Myth Takes,Heart of Hearts,Pieces Of The People We Love,The Rapture
1,!!!,"Strange Weather, Isn't It?",Hollow,Donkey,Cansei de Ser Sexy
2,1990s,Cookies,You're Supposed To Be My Friend,Little Death,Pete and the Pirates
3,1990s,Kicks,I Don't Even Know What That Is,"Right Thoughts, Right Words, Right Action (Del...",Franz Ferdinand
4,2562,Unbalance,Flashback,Great Lengths,Martyn
5,2:54,2:54,Creeping,Oshin,DIIV
6,2Pac,Pac's Life,Don't Sleep,Thug Life: Volume 1,Thug Life
7,50 Cent,Before I Self Destruct,Do You Think About Me,The Hunger for More (Deluxe Explicit Version),Lloyd Banks
8,50 Cent,Curtis,Peep Show,The County Hound EP,Ca$his
9,50 Cent,Get Rich or Die Tryin',Gotta Make It To Heaven,Beg For Mercy,G-Unit


## Recommendation EDA

Exploratory analysis of Last.fm batch results on `albums_2plus.csv`.

**System:** strategy **(b) random seed track**, `track.getSimilar` -> parent album, same-artist exclusion, 1 recommendation per seed album.

Run after the batch cell (cell 5). Requires `work` in memory or reloads from `WORK_PATH`.

In [ ]:
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CATALOG_PATH = DATA_DIR / "album_catalog_enriched.parquet"
HIDDEN_GEMS_PATH = DATA_DIR / "lastfm_hidden_gems.csv"

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 4)


def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):
        s = s.replace(ch, "'")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s)


def pair_key(artist, album):
    return f"{normalize(artist)}::{normalize(album)}"


def gini(counts):
    x = np.sort(np.asarray(counts, dtype=float))
    if x.size == 0 or x.sum() == 0:
        return np.nan
    n = x.size
    idx = np.arange(1, n + 1)
    return (2 * (idx * x).sum()) / (n * x.sum()) - (n + 1) / n


def top_n_share(counts, n, total):
    return counts.head(min(n, len(counts))).sum() / total


def collect_recs(df, n_recs):
    rows = []
    ok = df[df["status"] == "ok"]
    for _, row in ok.iterrows():
        seed_key = pair_key(row["artist"], row["album"])
        for rank in range(1, n_recs + 1):
            rec_album = row.get(f"rec_{rank}_album")
            rec_artist = row.get(f"rec_{rank}_artist")
            rec_score = row.get(f"rec_{rank}_score")
            if pd.isna(rec_album) or pd.isna(rec_artist):
                continue
            rows.append(
                {
                    "album_id": row["album_id"],
                    "seed_key": seed_key,
                    "seed_artist": row["artist"],
                    "seed_album": row["album"],
                    "review_count": row.get("review_count"),
                    "rec_key": pair_key(rec_artist, rec_album),
                    "rec_artist": rec_artist,
                    "rec_album": rec_album,
                    "similarity_score": rec_score,
                    "rank": rank,
                }
            )
    return pd.DataFrame(rows)


# reload if batch cell was skipped this session
if "work" not in globals():
    work = pd.read_csv(WORK_PATH)

recs = collect_recs(work, N_RECS)
bench_keys = {pair_key(a, b) for a, b in zip(work["artist"], work["album"])}
recs["rec_in_bench"] = recs["rec_key"].isin(bench_keys)

if CATALOG_PATH.exists():
    catalog = pd.read_parquet(CATALOG_PATH)
    catalog["pair_key"] = catalog["artist_norm"] + "::" + catalog["album_norm"]
    genre_lookup = (
        catalog.dropna(subset=["genre_pitchfork"])
        .drop_duplicates("pair_key")
        .set_index("pair_key")["genre_pitchfork"]
    )
    rating_lookup = (
        catalog.dropna(subset=["rating_mean"])
        .drop_duplicates("pair_key")
        .set_index("pair_key")["rating_mean"]
    )
    recs["seed_genre"] = recs["seed_key"].map(genre_lookup)
    recs["rec_genre"] = recs["rec_key"].map(genre_lookup)
    recs["seed_rating"] = recs["seed_key"].map(rating_lookup)
else:
    recs["seed_genre"] = pd.NA
    recs["rec_genre"] = pd.NA
    recs["seed_rating"] = pd.NA

n_recs = len(recs)
n_seeds = len(work)
n_ok = (work["status"] == "ok").sum()
print(f"Bench albums: {n_seeds:,}")
print(f"Successful recommendations: {n_ok:,} ({n_ok / n_seeds:.1%})")
print(f"Recommendation rows: {n_recs:,}")

In [ ]:
# --- 1. Score distribution ---
scores = recs["similarity_score"].dropna()
print("1. Score distribution")
print(scores.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(4))
print(f"   >0.75: {(scores > 0.75).mean():.1%}   >0.50: {(scores > 0.50).mean():.1%}")

fig, ax = plt.subplots()
ax.hist(scores, bins=40, color="#4472C4", edgecolor="white")
ax.set(xlabel="Last.fm track similarity score", ylabel="Count", title="Recommendation score distribution")
ax.axvline(scores.mean(), color="crimson", ls="--", label=f"mean={scores.mean():.3f}")
ax.legend()
plt.show()

# --- 2. Hubness ---
album_counts = recs["rec_key"].value_counts()
artist_counts = recs["rec_artist"].str.strip().str.lower().value_counts()
print("\n2. Hubness")
for k in [10, 50, 100, 500]:
    print(f"   Top {k:>3} albums account for {top_n_share(album_counts, k, n_recs):.1%} of recommendations")
print(f"   Gini (album concentration): {gini(album_counts.values):.3f}")
print(f"   Albums recommended >=5 times: {(album_counts >= 5).sum()}")
print("\n   Top hub albums:")
display(
    album_counts.head(15)
    .rename("query_count")
    .to_frame()
    .assign(
        artist=lambda df: [k.split("::", 1)[0] for k in df.index],
        album=lambda df: [k.split("::", 1)[1] for k in df.index],
    )[["artist", "album", "query_count"]]
)

fig, ax = plt.subplots()
ax.plot(range(1, len(album_counts) + 1), album_counts.values.cumsum() / n_recs, color="#ED7D31")
ax.set(xlabel="Album rank (by frequency)", ylabel="Cumulative share of recommendations", title="Hubness curve")
ax.axhline(0.5, color="gray", ls=":", lw=1)
plt.show()

In [ ]:
# --- 3. Coverage ---
unique_rec_albums = album_counts.size
unique_rec_artists = artist_counts.size
seeds_with_recs = recs["seed_key"].nunique()
never_recommended = (work["status"] == "no_results").sum()

print("3. Coverage")
print(f"   Seeds with a recommendation: {seeds_with_recs:,} / {n_seeds:,} ({seeds_with_recs / n_seeds:.1%})")
print(f"   Seeds with no Last.fm match: {never_recommended:,} ({never_recommended / n_seeds:.1%})")
print(f"   Unique recommended albums: {unique_rec_albums:,}")
print(f"   Unique recommended artists: {unique_rec_artists:,}")
print(f"   Album coverage (unique / total recs): {unique_rec_albums / n_recs:.1%}")
print(f"   Recs pointing back into bench catalog: {recs['rec_in_bench'].mean():.1%}")
print(f"   Recs outside bench catalog (hidden gems pool): {(~recs['rec_in_bench']).sum():,}")

# --- 4. Rating bias ---
rated = recs.dropna(subset=["seed_rating", "similarity_score"])
rating_corr = rated["seed_rating"].corr(rated["similarity_score"])
print("\n4. Rating bias (seed album mean critic rating vs Last.fm score)")
print(f"   Rows with seed rating: {len(rated):,} / {n_recs:,}")
print(f"   Correlation: {rating_corr:.4f}")
print(f"   Mean seed rating: {rated['seed_rating'].mean():.3f}")
print(f"   Mean score when seed rating >=4: {rated.loc[rated['seed_rating'] >= 4, 'similarity_score'].mean():.3f}")
print(f"   Mean score when seed rating <=2: {rated.loc[rated['seed_rating'] <= 2, 'similarity_score'].mean():.3f}")

review_corr = recs["review_count"].corr(recs["similarity_score"])
print(f"   Correlation with review_count: {review_corr:.4f}")

In [ ]:
# --- 5. Cross-genre clustering (Pitchfork genre tags, catalog-matched rows only) ---
genre_rows = recs.dropna(subset=["seed_genre", "rec_genre"])
within_genre = (genre_rows["seed_genre"] == genre_rows["rec_genre"]).mean()
base = genre_rows["rec_genre"].value_counts(normalize=True)

print("5. Cross-genre clustering")
print(f"   Rows with both seed and rec genre tags: {len(genre_rows):,} / {n_recs:,}")
print(f"   Within-genre recommendation rate: {within_genre:.1%}")

genre_lift = []
for genre, sub in genre_rows.groupby("seed_genre"):
    if len(sub) < 20:
        continue
    within = (sub["rec_genre"] == genre).mean()
    chance = base.get(genre, 0)
    if chance == 0:
        continue
    genre_lift.append(
        {
            "genre": genre,
            "n_seeds": len(sub),
            "within_rate": within,
            "base_rate": chance,
            "lift": within / chance,
        }
    )

genre_lift = pd.DataFrame(genre_lift).sort_values("lift", ascending=False)
display(genre_lift.round(3))

# --- 6. Reciprocity ---
edges = dict(zip(recs["seed_key"], recs["rec_key"]))
reciprocal = sum(1 for seed, rec in edges.items() if edges.get(rec) == seed)
print("\n6. Reciprocity")
print(f"   If A recommends B, B recommends A back: {reciprocal / len(edges):.1%} ({reciprocal:,} / {len(edges):,})")

# --- 7. Hidden gems (recs outside the bench catalog) ---
outside = recs.loc[~recs["rec_in_bench"]].copy()
hidden = (
    outside.groupby(["rec_artist", "rec_album", "rec_key"], as_index=False)
    .agg(
        query_count=("seed_key", "count"),
        mean_score=("similarity_score", "mean"),
        example_seed_artist=("seed_artist", "first"),
        example_seed_album=("seed_album", "first"),
    )
    .sort_values(["query_count", "mean_score"], ascending=[False, False])
)

print("\n7. Hidden gems (recommended albums not in bench set)")
print(f"   Unique hidden-gem albums: {len(hidden):,}")
print(f"   Hidden gems recommended >=3 times: {(hidden['query_count'] >= 3).sum():,}")
display(hidden.head(20))

hidden.to_csv(HIDDEN_GEMS_PATH, index=False)
print(f"\nSaved: {HIDDEN_GEMS_PATH}")

In [ ]:
# --- 8. Novelty, diversity, serendipity ---
popularity = recs["rec_key"].map(album_counts) / n_recs
recs["novelty"] = -np.log2(popularity)
recs["serendipity"] = recs["similarity_score"] * (1 - popularity)
album_probs = album_counts / n_recs

print("8. Novelty / diversity / serendipity")
print(f"   Mean novelty (-log2 freq): {recs['novelty'].mean():.3f}")
print(f"   Mean serendipity (score x (1-popularity)): {recs['serendipity'].mean():.3f}")
print(f"   Album entropy (bits): {-(album_probs * np.log2(album_probs)).sum():.3f}")

print("\n   Highest serendipity picks:")
display(
    recs.nlargest(10, "serendipity")[
        ["seed_artist", "seed_album", "rec_artist", "rec_album", "similarity_score", "novelty", "serendipity"]
    ]
)

## EDA Summary

**System:** Last.fm track-similarity recommender, strategy **(b) random seed track**, 1 rec per bench album (`albums_2plus.csv`, 2,498 albums from the critic review corpus).

Re-run the analysis cells above after batch completion to refresh these numbers.

### Key findings

1. **Score distribution** — Last.fm similarity scores are much lower and wider than embedding cosine scores (embedding mean ~0.80; Last.fm mean ~0.24). Scores span nearly the full 0–1 range; this pipeline is not score-compressed like the review-embedding recommender.

2. **Hubness** — Moderate concentration: top 50 albums account for ~8% of recommendations (embedding PF-only: top 500 at ~22%). Gini ~0.11. Frequent hubs are canonical indie/electronic albums (*The Bends*, *Tourist History*, *Superunknown*) surfaced by many seed tracks.

3. **Coverage** — ~85% of bench albums get a recommendation; ~12% return `no_results`. ~88% of recommended albums are unique. ~78% of recommendations point **outside** the bench catalog — Last.fm pulls from the broader listening graph, not just reviewed albums.

4. **Rating bias** — Near-zero correlation between seed album critic rating and Last.fm score (r ~ −0.07). Review corpus popularity does not predict Last.fm match strength.

5. **Cross-genre clustering** — On catalog-matched rows (~38% of recs have Pitchfork genre tags for both seed and rec), within-genre rate ~62%. Lift is highest for Rap (~28x) and Electronic (~5.5x), lowest for Rock (~1.4x) — same qualitative pattern as embedding analysis, on a smaller tagged subset.

6. **Reciprocity** — Very low (~1.7%) vs ~26–29% for review embeddings. Track-similarity recommendations are asymmetric.

7. **Hidden gems** — ~1,540 unique albums outside the bench set. Top recurring picks are well-known adjacent-scene albums, not obscure deep cuts. Exported to `lastfm_hidden_gems.csv`.

8. **Novelty / serendipity** — High novelty (mean ~10.8 bits; most albums recommended once). Mean serendipity ~0.24.